In [1]:
from __future__ import annotations

import json
import os
import random
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"项目目录: {PROJECT_ROOT.resolve()}")

项目目录: D:\CodeData\Program Coding\Project\Writing_Coach_Agent


In [2]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


@dataclass
class PlanStep:
    step_id: str
    tool_name: str
    purpose: str
    inputs: dict[str, Any]
    status: StepStatus = StepStatus.PENDING
    result: Any = None
    error: Optional[str] = None


@dataclass
class AgentState:
    task: str
    inputs: dict[str, Any]
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    plan: list[PlanStep] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    final_answer: Optional[dict[str, Any]] = None

    def log(self, event: str, **payload: Any) -> None:
        self.trace.append({
            "time": time.strftime("%H:%M:%S"),
            "run_id": self.run_id,
            "event": event,
            **payload,
        })


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, func: Callable[..., Any]) -> None:
        if name in self._tools:
            raise ValueError(f"工具已注册: {name}")
        self._tools[name] = func

    def call(self, name: str, **kwargs: Any) -> Any:
        if name not in self._tools:
            raise KeyError(f"未知工具: {name}")
        return self._tools[name](**kwargs)

    @property
    def names(self) -> list[str]:
        return sorted(self._tools)

In [3]:
class RetryableToolError(RuntimeError):
    """临时性错误：重试后可能恢复。"""


class FatalToolError(RuntimeError):
    """不可恢复错误：应停止或重新规划。"""


@dataclass
class Memory:
    working: dict[str, Any] = field(default_factory=dict)
    episodic: list[dict[str, Any]] = field(default_factory=list)

    def remember_result(self, step: PlanStep) -> None:
        self.working[step.step_id] = step.result
        self.episodic.append({"step_id": step.step_id, "tool": step.tool_name, "status": step.status.value})


class CheckpointStore:
    def __init__(self, folder: Path) -> None:
        self.folder = folder
        self.folder.mkdir(parents=True, exist_ok=True)

    def save(self, state: AgentState, memory: Memory) -> Path:
        path = self.folder / f"{state.run_id}.json"
        payload = {
            "run_id": state.run_id,
            "task": state.task,
            "inputs": state.inputs,
            "artifacts": state.artifacts,
            "trace": state.trace,
            "memory": asdict(memory),
            "plan": [asdict(step) for step in state.plan],
        }
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        return path

In [4]:
class FlakyEvidenceTool:
    def __init__(self, fail_times: int = 1) -> None:
        self.remaining_failures = fail_times

    def __call__(self, essay: str) -> dict[str, Any]:
        if self.remaining_failures > 0:
            self.remaining_failures -= 1
            raise RetryableToolError("模拟的模型服务 503")
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", essay) if s.strip()]
        return {"evidence": sentences[:2], "quality": "ok" if len(sentences) >= 2 else "weak"}


def validate_input_tool(essay: str) -> dict[str, Any]:
    if not isinstance(essay, str) or not essay.strip():
        raise FatalToolError("作文为空")
    return {"essay": re.sub(r"\s+", " ", essay).strip()}


def score_tool(essay: str) -> dict[str, Any]:
    words = re.findall(r"\b[A-Za-z']+\b", essay)
    markers = sum(bool(re.search(rf"\b{x}\b", essay, re.I)) for x in ["because", "however", "therefore", "example"])
    return {"language": round(min(5, 1.5 + len(words) / 45), 2), "argumentation": round(min(5, 1.2 + markers * 0.75), 2)}


def report_tool(scores: dict[str, Any], evidence: dict[str, Any]) -> dict[str, Any]:
    return {"scores": scores, "evidence": evidence["evidence"], "status": "completed"}

In [5]:
@dataclass
class Reflection:
    action: str  # retry / replan / continue / stop
    reason: str


class Reflector:
    def review_failure(self, step: PlanStep, attempt: int, max_retries: int) -> Reflection:
        if step.error and "RetryableToolError" in step.error and attempt < max_retries:
            return Reflection("retry", "临时性错误且仍有重试预算")
        if step.tool_name == "extract_evidence":
            return Reflection("replan", "证据工具不可用，改用轻量降级工具")
        return Reflection("stop", "错误不可恢复或超出预算")

    def review_result(self, step: PlanStep) -> Reflection:
        if step.tool_name == "extract_evidence" and step.result.get("quality") == "weak":
            return Reflection("continue", "证据较弱，但仍可生成带风险提示的报告")
        return Reflection("continue", "结果通过基本检查")


def should_stop(state: AgentState, total_steps: int, max_steps: int = 10) -> tuple[bool, str]:
    if len([x for x in state.trace if x["event"] == "tool_started"]) >= max_steps:
        return True, "达到最大工具调用次数"
    if state.final_answer is not None:
        return True, "已产生最终答案"
    if all(s.status == StepStatus.SUCCEEDED for s in state.plan[:total_steps]):
        return False, "当前计划已完成，等待组装答案"
    return False, "继续执行"